# `code/pipeline/p001_07_robustness_grid.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_07 강건성 격자 (W4) — 위협별 축을 두 핵심 estimand 에 적용 (동결 grid 준수)

[대상 estimand] (기준값, NA+EU)
  A = E1 섹터층 계수 (ff~fp | 투자사×연도×섹터): 기준 +3.21 [1.96, 4.75] — 논문의 유의 매칭층
  B = E2 후속 판별 (fon~fp | 투자사×연도×섹터×스테이지, ff=1 딜): 기준 +1.08 [−1.48, +3.62]
[행 = 위협 → 축] (각 행은 기준에서 한 가지만 변경 — 레버 비축적)
  T1 측정(ff 정의 과대): ff=다수 여성 / ff=단독 여성 창업자
  T2 귀속 노이즈(R2): 단독 귀속 파트너 라운드만
  T3 창 선택: fon 24m (B 만)
  T4 성별 측정(R1/KQ4 — CB 에 신뢰도 필드 부재): 프로필 충실 표본(학위 ≥1 또는 jobs ≥2 인
     인물의 성별만 판정) — 대리 정의, 문서화
  T5 추론: 군집 = 파트너 / 기업 (기준 = 투자사)
[사전 예측] (결과 전, 2026-09-03)
  P1 A 는 전 행에서 동부호·CI 0 배제 (기준 +3.21 의 절반 이상).
  P2 B 는 전 행에서 하한 > −5pp (호의 배제 유지).
  P3 T4 프로필 충실 표본에서 계수 크기 유지·유의 (성별 오분류가 결과를 만들지 않음).
[판정] P1·P2 전 행 통과 → GO. 특정 행 실패 → 해당 위협을 claim ceiling 에 반영해 정직 기록
  (실패 행도 negative_results 와 매트릭스에 보존).
```


In [ ]:
import os
import sys

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
os.environ["HARNESS_OUT"] = os.environ.get("P001_OUT", os.path.join(HERE, "out"))  # set P001_* / CRUNCHBASE_RAW to your local copies (licensed inputs; see DATA_ACCESS.md)
sys.path.insert(0, HERE)  # gates.py / emit_contract.py sit alongside in code/pipeline
from emit_contract import emit, qci  # noqa: E402
from gates import CTX  # noqa: E402

rng = np.random.default_rng(42)
NB = 400
EU = {"GBR", "DEU", "FRA", "NLD", "SWE", "ESP", "ITA", "CHE", "BEL", "AUT", "DNK", "FIN", "NOR",
      "IRL", "PRT", "POL", "CZE", "EST", "LTU", "LVA", "GRC", "HUN", "ROU", "LUX"}
NAEU = EU | {"USA", "CAN"}

d0 = pd.read_parquet(os.environ.get("P001_SAMPLE", "/path/to/sample_v1.parquet"))
d0 = d0[d0["country_code"].isin(NAEU)].copy()
d0["dt"] = pd.to_datetime(d0["dt"])

# 보조표 (문서화): 단독 여성 창업 / 프로필 충실 성별 / fon24
people = CTX.people[["uuid", "gender"]]
g_map = people[people["gender"].isin(["male", "female"])].set_index("uuid")["gender"]
jobs = CTX.jobs
fj = jobs[jobs["title"].fillna("").str.lower().str.contains("founder", regex=False)][
    ["person_uuid", "org_uuid"]].dropna()
fj["fg"] = fj["person_uuid"].map(g_map)
fj = fj[fj["fg"].notna()]
grp = fj.groupby("org_uuid")["fg"]
org_solo_f = (grp.size() == 1) & (grp.first() == "female")
deg_p = set(CTX.degrees.dropna(subset=["person_uuid"])["person_uuid"])
jcnt = jobs.groupby("person_uuid").size()
rich = set(jcnt[jcnt >= 2].index) | deg_p
fj_rich = fj[fj["person_uuid"].isin(rich)]
org_ff_rich = (fj_rich["fg"] == "female").groupby(fj_rich["org_uuid"]).max()
pt_rich_gender = {p: g for p, g in g_map.items() if p in rich}  # noqa: F841 (파트너측은 아래 map)

rounds = CTX.rounds.dropna(subset=["announced_on", "org_uuid"]).copy()
rounds["rdt"] = pd.to_datetime(rounds["announced_on"], errors="coerce")
r_org = rounds[["org_uuid", "rdt"]].dropna().sort_values(["org_uuid", "rdt"])
r_org["next_dt"] = r_org.groupby("org_uuid")["rdt"].shift(-1)
next_map = r_org.drop_duplicates(["org_uuid", "rdt"]).set_index(["org_uuid", "rdt"])["next_dt"]
d0["_next"] = pd.Series(list(zip(d0["org_uuid"], d0["dt"]))).map(next_map).to_numpy()
d0["fon24"] = ((pd.to_datetime(d0["_next"]) - d0["dt"]).dt.days <= 365 * 2).fillna(False).astype(float)

d0["ff_solo"] = d0["org_uuid"].map(org_solo_f).fillna(False).astype(float)
d0["ff_rich"] = d0["org_uuid"].map(org_ff_rich)
d0["p_rich"] = d0["partner_uuid"].isin(rich)


def fwl(df, y, x, cell, cl, nb=NB):
    dd = df[[y, x, cell, cl]].dropna(subset=[y, x]).reset_index(drop=True)
    yr = (dd[y].astype(float) - dd[y].astype(float).groupby(dd[cell]).transform("mean")).to_numpy()
    xr = (dd[x].astype(float) - dd[x].astype(float).groupby(dd[cell]).transform("mean")).to_numpy()
    sxx = (xr * xr).sum()
    if sxx == 0 or len(dd) < 500:
        return float("nan"), [float("nan")] * 2, int(len(dd))
    beta = float((xr * yr).sum() / sxx)
    g_ = {c: g.index.to_numpy() for c, g in dd.groupby(cl)}
    keys = list(g_)
    bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(keys), len(keys))
        rows_ = np.concatenate([g_[keys[i]] for i in pick])
        x2, y2 = xr[rows_], yr[rows_]
        s = (x2 * x2).sum()
        if s:
            bs.append((x2 * y2).sum() / s)
    return round(beta, 4), qci(bs), int(len(dd))


rowsA, rowsB = {}, {}
dB = d0[(d0["ff"] == 1.0) & (d0["dt"] <= "2020-10-31")]
rowsA["baseline"] = fwl(d0, "ff", "fp", "cell_cat", "investor_uuid")
rowsB["baseline"] = fwl(dB, "fon", "fp", "cell_stage", "investor_uuid")
# T1 ff 정의
rowsA["T1_ff_majority"] = fwl(d0, "ffm", "fp", "cell_cat", "investor_uuid")
rowsA["T1_ff_solo"] = fwl(d0, "ff_solo", "fp", "cell_cat", "investor_uuid")
dB_m = d0[(d0["ffm"] == 1.0) & (d0["dt"] <= "2020-10-31")]
rowsB["T1_ff_majority"] = fwl(dB_m, "fon", "fp", "cell_stage", "investor_uuid")
# T2 단독 귀속
rowsA["T2_solo_attr"] = fwl(d0[d0["solo_attr"]], "ff", "fp", "cell_cat", "investor_uuid")
rowsB["T2_solo_attr"] = fwl(dB[dB["solo_attr"]], "fon", "fp", "cell_stage", "investor_uuid")
# T3 창
rowsB["T3_fon24"] = fwl(d0[(d0["ff"] == 1.0) & (d0["dt"] <= "2021-10-31")], "fon24", "fp",
                        "cell_stage", "investor_uuid")
# T4 프로필 충실 (파트너·창업자 양측)
d4 = d0[d0["p_rich"] & d0["ff_rich"].notna()].copy()
d4["ffr"] = d4["ff_rich"].astype(float)
rowsA["T4_profile_rich"] = fwl(d4, "ffr", "fp", "cell_cat", "investor_uuid")
d4B = d4[(d4["ffr"] == 1.0) & (d4["dt"] <= "2020-10-31")]
rowsB["T4_profile_rich"] = fwl(d4B, "fon", "fp", "cell_stage", "investor_uuid")
# T5 군집
rowsA["T5_cluster_partner"] = fwl(d0, "ff", "fp", "cell_cat", "partner_uuid")
rowsA["T5_cluster_org"] = fwl(d0, "ff", "fp", "cell_cat", "org_uuid")
rowsB["T5_cluster_partner"] = fwl(dB, "fon", "fp", "cell_stage", "partner_uuid")

a_pass = all((not np.isnan(v[0])) and v[1][0] > 0 and v[0] >= 0.5 * rowsA["baseline"][0]
             for k, v in rowsA.items() if not k.startswith("T1_ff_solo"))
a_solo_sign = rowsA["T1_ff_solo"][0] > 0
b_pass = all((not np.isnan(v[0])) and v[1][0] > -0.05 for v in rowsB.values())
status = "GO" if (a_pass and b_pass) else "PARTIAL"
fails = [k for k, v in rowsA.items() if not (v[1][0] > 0)] + \
        [f"B:{k}" for k, v in rowsB.items() if not (v[1][0] > -0.05)]
verdict = (f"A(섹터 매칭층) {len(rowsA)}행: 기준 {rowsA['baseline'][0]:+.4f} — "
           + ("전 행 유의·절반 이상 유지" if a_pass else f"실패 행 {fails}")
           + f" (solo 정의 부호 {'+' if a_solo_sign else '−'}); "
           f"B(판별 fon) {len(rowsB)}행: " + ("전 행 하한 > −5pp — 호의 배제 격자 전체 유지" if b_pass
                                            else f"실패 {fails}"))

emit("P001-07", "강건성 격자 — 측정·표본·창·추론 축 (W4)", status,
     {"A_e1_sector": {k: [v[0], v[1], v[2]] for k, v in rowsA.items()},
      "B_e2_fon": {k: [v[0], v[1], v[2]] for k, v in rowsB.items()},
      "a_all_pass": bool(a_pass), "b_all_pass": bool(b_pass), "fails": fails},
     prediction="A 전 행 CI>0·기준의 절반 이상; B 전 행 하한 > −5pp; 프로필 충실 표본 유지",
     verdict=verdict, kill_met=False, n=rowsA["baseline"][2],
     extra={"stage": 4, "feeds": "ROBUSTNESS_MATRIX.md", "slug": "robustness_grid",
            "inputs": "sample_v1 + 보조표(solo/rich/fon24 — cores_v1)"})
print("done")
